In [ ]:
!pip install -q kagglehub torch torchvision timm opencv-python matplotlib gradio

import os, time, copy, random, numpy as np, cv2
import matplotlib.pyplot as plt
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
from PIL import Image
import gradio as gr

from google.colab import drive
drive.mount('/content/drive')

# ---------------- GPU SETTINGS (T4 OPTIMIZED) ----------------
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ---------------- CONFIG ----------------
img_size = 224
BATCH_SIZE = 16
NUM_EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "swin_tiny_patch4_window7_224"

SAVE_DIR = "/content/drive/MyDrive/Final/output"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, "best_swin_96plus.pth")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---------------- DATASET ----------------
print("Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("gunavenkatdoddi/eye-diseases-classification")

DATA_DIR = os.path.join(path, "dataset")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = path

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
NUM_CLASSES = len(class_names)

total_len = len(base_dataset)
train_len = int(0.7 * total_len)
val_len   = int(0.15 * total_len)
test_len  = total_len - train_len - val_len

train_base, val_base, test_base = random_split(
    base_dataset,
    [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(42)
)

class DatasetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        img = self.transform(img)
        return img, label

train_dataset = DatasetWithTransform(train_base, train_transform)
val_dataset   = DatasetWithTransform(val_base, val_test_transform)
test_dataset  = DatasetWithTransform(test_base, val_test_transform)

targets = [train_base[i][1] for i in range(len(train_base))]
class_count = np.bincount(targets)
weights = 1. / class_count
sample_weights = [weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

# ---------------- MODEL ----------------
model = timm.create_model(MODEL_NAME, pretrained=True)
model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
best_acc = 0.0
best_wts = copy.deepcopy(model.state_dict())

# ---------------- TRAINING ----------------
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    for phase, loader in [("train", train_loader), ("val", val_loader)]:
        model.train() if phase == "train" else model.eval()

        running_loss, running_corrects, total = 0.0, 0, 0

        for inputs, labels in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == "train"):
                outputs = model(inputs)
                if outputs.ndim > 2:
                    outputs = outputs.mean(dim=list(range(2, outputs.ndim)))
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                if phase == "train":
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels)
            total += inputs.size(0)

        epoch_loss = running_loss / total
        epoch_acc = (running_corrects.double() / total).item()
        print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if phase == "train":
            history["train_loss"].append(epoch_loss)
            history["train_acc"].append(epoch_acc)
        else:
            history["val_loss"].append(epoch_loss)
            history["val_acc"].append(epoch_acc)
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())
                torch.save(best_wts, SAVE_PATH)
                print(f"✅ Model saved to {SAVE_PATH}")

    scheduler.step()

# ---------------- TEST ----------------
model.load_state_dict(best_wts)
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        outputs = model(inputs)
        if outputs.ndim > 2:
            outputs = outputs.mean(dim=list(range(2, outputs.ndim)))
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

test_acc = correct / total
print(f"\n🎯 FINAL TEST ACCURACY: {test_acc:.4f}")

# ---------------- GRADIO ----------------
def predict_image(image):
    image = image.convert("RGB").resize((img_size, img_size))
    t = val_test_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(t)
        if out.ndim > 2:
            out = out.mean(dim=list(range(2, out.ndim)))
        probs = torch.softmax(out, dim=1)[0]
        idx = probs.argmax().item()
    return f"Predicted: {class_names[idx]} | Confidence: {probs[idx]:.3f} | Test Acc: {test_acc:.4f}"

with gr.Blocks() as app:
    gr.Markdown("## 🩺 Eye Disease Detection (Swin Transformer)")
    img = gr.Image(type="pil")
    out = gr.Textbox()
    gr.Button("Predict").click(predict_image, img, out)

app.launch()


In [ ]:
# ===================== INSTALLS =====================
!pip install -q scikit-learn seaborn timm kagglehub

# ===================== IMPORTS =====================
import os, torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ===================== DEVICE =====================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ===================== PATHS =====================
MODEL_PATH = "/content/drive/MyDrive/Final/output/best_swin_96plus.pth"

import kagglehub
DATA_PATH = kagglehub.dataset_download("gunavenkatdoddi/eye-diseases-classification")
DATA_DIR = os.path.join(DATA_PATH, "dataset")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = DATA_PATH

# ===================== TRANSFORMS =====================
img_size = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ===================== DATASET =====================
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
class_names = dataset.classes
NUM_CLASSES = len(class_names)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

# ===================== LOAD MODEL =====================
model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False)
model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("Model loaded successfully")

# ===================== INFERENCE =====================
y_true, y_pred, y_prob, features = [], [], [], []

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        if outputs.ndim > 2:
            outputs = outputs.mean(dim=list(range(2, outputs.ndim)))
        probs = torch.softmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(torch.argmax(probs, 1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())
        features.extend(outputs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)
features = np.array(features)

# ===================== METRICS =====================
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted")
rec = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print("\n===== EVALUATION METRICS =====")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

# ===================== CONFUSION MATRIX =====================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ===================== ROC & AUC =====================
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
fpr, tpr, roc_auc = {}, {}, {}

for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(7,6))
for i in range(NUM_CLASSES):
    plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-Class ROC Curve")
plt.legend()
plt.grid()

plt.show()

# ===================== t-SNE =====================
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_features = tsne.fit_transform(features)

plt.figure(figsize=(7,6))
for i, cls in enumerate(class_names):
    idx = y_true == i
    plt.scatter(tsne_features[idx,0], tsne_features[idx,1], label=cls, s=8)
plt.legend()
plt.title("t-SNE Feature Visualization")
plt.show()

# ===================== HARD-CODED IMAGE PREDICTIONS =====================
print("\n===== SAMPLE PREDICTIONS =====")

sample_images = []
for cls in class_names[:4]:
    cls_dir = os.path.join(DATA_DIR, cls)
    img_name = os.listdir(cls_dir)[0]
    sample_images.append(os.path.join(cls_dir, img_name))

plt.figure(figsize=(12,4))

for i, img_path in enumerate(sample_images):
    img = Image.open(img_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(input_tensor)
        if out.ndim > 2:
            out = out.mean(dim=list(range(2, out.ndim)))
        prob = torch.softmax(out, dim=1)[0]
        idx = prob.argmax().item()

    plt.subplot(1,4,i+1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{class_names[idx]}\nConf: {prob[idx]:.2f}")

plt.suptitle("Hard-Coded Image Predictions")
plt.show()
